# 2. 다변량 강건 회귀분석

## 요약

정상 배치 90개에서 전략과 주요 공정변수를 동시에 보정했다. 세 성과지표 모두 산 총투입량과 후기 기질 기울기만 일관되게 음의 관계를 보였고, 공정변수를 포함하면 APC 전략 더미는 유의하지 않았다. 다만 이 변수들은 전략의 결과로 발생한 매개변수일 수 있으므로 인과효과로 해석하지 않는다. CSV는 저장하지 않는다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats


def find_project_root(start=Path.cwd()):
    for root in (start, *start.parents):
        if (root / 'data/interim/merged_data_ko.csv').exists():
            return root
    raise FileNotFoundError('merged_data_ko.csv를 찾을 수 없습니다.')


data = pd.read_csv(find_project_root() / 'data/interim/merged_data_ko.csv')
data = data.loc[data['배치번호'] <= 90].copy()
print(f'정상 배치 {data["배치번호"].nunique()}개')

정상 배치 90개


### 판단

Fault의 극단적 패턴을 제외한 정상 배치만 사용한다. 표본 단위는 시계열 행이 아니라 배치이며, RC를 전략 기준범주로 둔다.

In [2]:
rows = []
for batch_number, batch in data.groupby('배치번호', sort=True):
    batch = batch.sort_values('발효시간(h)')
    time = batch['발효시간(h)'].to_numpy()
    late = time / time[-1] >= 0.8
    penicillin = batch['페니실린농도(g/L)'].to_numpy()
    substrate = batch['기질농도(g/L)'].to_numpy()
    rows.append({
        '배치번호': batch_number,
        '전략': 'RC' if batch_number <= 30 else ('OC' if batch_number <= 60 else 'APC'),
        '최종농도': penicillin[-1],
        '농도유지율': penicillin[-1] / penicillin.max(),
        '생산성': penicillin[-1] / time[-1],
        '산총투입량': np.trapezoid(batch['산투입유량(L/h)'], time),
        '기질후기기울기': np.polyfit(time[late], substrate[late], 1)[0],
        'OUR평균': batch['산소소모율(g/min)'].mean(),
        'pH표준편차': batch['pH'].std(ddof=1),
        'DO평균': batch['용존산소(mg/L)'].mean(),
    })
batch_metrics = pd.DataFrame(rows)
display(batch_metrics.groupby('전략').size().rename('배치수').to_frame())

,배치수
전략,
APC,30
OC,30
RC,30


### 판단

연속형 설명변수는 1표준편차 단위로 표준화한다. 따라서 계수는 다른 단위의 공정변수끼리도 영향 방향과 상대 크기를 비교할 수 있다.

In [3]:
features = ['산총투입량', '기질후기기울기', 'OUR평균', 'pH표준편차', 'DO평균']
standardized = (batch_metrics[features] - batch_metrics[features].mean()) / batch_metrics[features].std(ddof=1)
design = np.column_stack([
    np.ones(len(batch_metrics)),
    batch_metrics['전략'].eq('OC').astype(float),
    batch_metrics['전략'].eq('APC').astype(float),
    standardized.to_numpy(),
])
term_names = ['절편', 'OC-RC', 'APC-RC'] + features

vif_rows = []
for column_index, term in enumerate(term_names[1:], start=1):
    target = design[:, column_index]
    other_columns = np.delete(design, column_index, axis=1)
    fitted = other_columns @ (np.linalg.pinv(other_columns) @ target)
    r_squared = 1 - np.sum((target - fitted) ** 2) / np.sum((target - target.mean()) ** 2)
    vif_rows.append({'변수': term, 'VIF': 1 / (1 - r_squared)})
vif_table = pd.DataFrame(vif_rows)
display(vif_table.round(3))

,변수,VIF
0,OC-RC,1.443
1,APC-RC,1.580
2,산총투입량,6.527
3,기질후기기울기,5.676
4,OUR평균,2.568
5,pH표준편차,1.099
6,DO평균,1.410


### 판단

산 총투입량 VIF는 6.53, 후기 기질 기울기는 5.68로 중간 이상의 다중공선성이 있다. 계수 부호는 해석할 수 있지만 각 계수의 크기를 독립적인 원인효과로 단정하면 안 된다.

In [4]:
def fit_ols_hc3(x, y):
    coefficients = np.linalg.pinv(x) @ y
    residuals = y - x @ coefficients
    n_rows, n_parameters = x.shape
    inverse_xtx = np.linalg.pinv(x.T @ x)
    leverage = np.sum(x * (x @ inverse_xtx), axis=1)
    meat = x.T @ (((residuals / (1 - leverage)) ** 2)[:, None] * x)
    covariance = inverse_xtx @ meat @ inverse_xtx
    standard_errors = np.sqrt(np.diag(covariance))
    degrees_freedom = n_rows - n_parameters
    t_values = coefficients / standard_errors
    p_values = 2 * stats.t.sf(abs(t_values), degrees_freedom)
    critical = stats.t.ppf(0.975, degrees_freedom)
    r_squared = 1 - np.sum(residuals ** 2) / np.sum((y - y.mean()) ** 2)
    adjusted_r_squared = 1 - (1 - r_squared) * (n_rows - 1) / degrees_freedom
    return coefficients, standard_errors, p_values, coefficients - critical * standard_errors, coefficients + critical * standard_errors, r_squared, adjusted_r_squared


rng = np.random.default_rng(42)
fold_ids = np.empty(len(batch_metrics), dtype=int)
for strategy in ['RC', 'OC', 'APC']:
    indices = np.flatnonzero(batch_metrics['전략'].eq(strategy).to_numpy())
    rng.shuffle(indices)
    fold_ids[indices] = np.arange(len(indices)) % 5

coefficient_tables = []
model_rows = []
for outcome in ['최종농도', '농도유지율', '생산성']:
    target = batch_metrics[outcome].to_numpy()
    coef, se, p_value, lower, upper, r_squared, adjusted_r_squared = fit_ols_hc3(design, target)
    predictions = np.zeros(len(target))
    for fold in range(5):
        train = fold_ids != fold
        test = ~train
        predictions[test] = design[test] @ (np.linalg.pinv(design[train]) @ target[train])
    cv_r_squared = 1 - np.sum((target - predictions) ** 2) / np.sum((target - target.mean()) ** 2)
    cv_rmse = np.sqrt(np.mean((target - predictions) ** 2))
    coefficient_tables.append(pd.DataFrame({
        '성과': outcome, '변수': term_names, '계수': coef, 'HC3_SE': se,
        '95%CI_하한': lower, '95%CI_상한': upper, 'p값': p_value,
    }))
    model_rows.append({'성과': outcome, 'R2': r_squared, '수정R2': adjusted_r_squared, '교차검증R2': cv_r_squared, '교차검증RMSE': cv_rmse})
coefficient_results = pd.concat(coefficient_tables, ignore_index=True)
model_results = pd.DataFrame(model_rows)
display(model_results.round(6))
display(coefficient_results.loc[coefficient_results['변수'].ne('절편')].round(6))

,성과,R2,수정R2,교차검증R2,교차검증RMSE
0,최종농도,0.895109,0.886154,0.855595,2.899989
1,농도유지율,0.967740,0.964986,0.956159,0.035711
2,생산성,0.902801,0.894503,0.879951,0.012209


,성과,변수,계수,HC3_SE,95%CI_하한,95%CI_상한,p값
1,최종농도,OC-RC,0.585573,0.863660,-1.132523,2.303668,0.499672
2,최종농도,APC-RC,-0.957879,0.687137,-2.324814,0.409056,0.167080
3,최종농도,산총투입량,-5.272403,0.843550,-6.950493,-3.594313,0.000000
4,최종농도,기질후기기울기,-2.470672,0.718498,-3.899993,-1.041351,0.000921
5,최종농도,OUR평균,-0.147721,0.506369,-1.155051,0.859609,0.771233
6,최종농도,pH표준편차,-0.710443,0.494012,-1.693190,0.272304,0.154209
7,최종농도,DO평균,-0.377110,0.676255,-1.722396,0.968176,0.578606
9,농도유지율,OC-RC,0.008831,0.012218,-0.015475,0.033137,0.471885
10,농도유지율,APC-RC,0.008171,0.005739,-0.003245,0.019587,0.158273
11,농도유지율,산총투입량,-0.111718,0.013688,-0.138949,-0.084488,0.000000


### 최종 판단

- 교차검증 R²는 최종농도 0.856, 농도유지율 0.956, 생산성 0.880으로 표본 내 재현성은 양호했다.
- 산 총투입량이 1표준편차 증가할 때 최종농도는 5.27g/L, 농도유지율은 0.112, 생산성은 0.0244g/L·h 감소하는 관계가 있었다. 세 결과 모두 p<0.001이다.
- 후기 기질 기울기도 세 성과 모두와 유의한 음의 관계였다. 후기 기질 축적은 산 투입량을 함께 고려해도 독립적인 저성과 후보 신호다.
- OUR 평균, pH 변동, DO 평균과 전략 더미는 다른 변수를 보정하면 유의하지 않았다. 특히 APC 더미가 사라진 것은 APC 효과가 없다는 증명이 아니라 APC의 운전 특성을 공정변수가 설명했을 가능성을 뜻한다.
- 산 투입량과 후기 기질 기울기는 배치가 진행된 뒤 결정되는 매개변수일 수 있다. 이 회귀는 설명·예측 모델이며 전략의 인과효과 추정 모델이 아니다.